# Production Agent Harness Architecture

Planner/generator/evaluator loops, typed state graphs with checkpoints, MCP decoupling, and isolated parallel subagents.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice. It maps directly onto how production
multi-agent harnesses are built (the "Arbor harness" shape): the state graph
is the spine, subagents are nodes, MCP provides the hands, checkpoints
provide durability, and evaluator gates provide quality control.

## Learning Objectives

- Explain the planner/generator/evaluator loop, how it differs from plain ReAct, and when the extra calls pay off.
- Describe typed state graphs (LangGraph-style): typed schemas, reducers, checkpointers, thread resume, interrupts, time-travel.
- Explain MCP's primitives and the Brain/Hands/Session decoupling pattern.
- Describe orchestrator-worker parallelism, why context *isolation* is the point, and the 15x token economics.
- Compose all of it into one production harness and explain what failure mode each layer addresses.

## Mental Model

Each layer of a production harness exists to compensate for one specific
LLM failure mode:

| Layer | Compensates for |
|---|---|
| Typed state graph | drift, un-auditable control flow |
| Checkpointed persistence | crashes, long-running human approvals |
| MCP decoupling (Brain/Hands/Session) | integration sprawl, credential leakage |
| Isolated parallel subagents | context window limits, context pollution |
| Evaluator gates | unreliable output quality |

One-line synthesis for the interview: *the graph gives control and
auditability, checkpoints give durability, MCP gives capability decoupling,
subagent isolation gives context scaling, and evaluator loops give quality
control.*

## Important Concepts

- **Evaluator-optimizer** (Anthropic's "Building Effective Agents" taxonomy): plan → generate → evaluate against a rubric → revise; one of five workflow patterns, distinct from open-ended agents.
- **ReAct vs P/G/E**: ReAct is *emergent* control flow in one context (the model that made the mistake judges it); P/G/E is *engineered* control flow with an independent evaluator context.
- **Typed state graph**: nodes return partial state updates; **reducers** define merge semantics (what makes parallel fan-out safe); conditional edges route on state.
- **Checkpointer**: full state snapshot every superstep, keyed by `thread_id` → crash recovery, `interrupt()`/resume for human-in-the-loop, time-travel forking, audit.
- **MCP primitives**: tools, resources, prompts (server→client); sampling, roots, elicitation (client→server); stdio and Streamable HTTP transports.
- **Brain/Hands/Session**: LLM reasoning (swap models freely) / MCP tool servers as microservices (own credentials, scaling, sandbox) / conversation state in the persistence layer (any brain resumes any session; hands stay stateless).
- **Orchestrator-worker**: lead agent plans, spawns 3-5 parallel subagents with explicit mandates, synthesizes structured results.

## Need-To-Know Coverage Checklist

- [x] P/G/E loop mechanics, termination criteria, and failure modes (rubber-stamping, oscillation, reward hacking).
- [x] LangGraph: StateGraph, typed schemas, reducers, PostgresSaver, thread_id resume, interrupt(), Send fan-out.
- [x] Checkpoints vs true durable execution (Temporal) — the side-effect replay caveat.
- [x] MCP spec status and adoption (OpenAI, Google); 2025-11-25 release (async tasks, URL-mode elicitation).
- [x] MCP security: trust boundaries, tool poisoning, rug pulls, roots scoping.
- [x] Anthropic multi-agent research system: 90.2% improvement, ~15x tokens, context isolation as the point.
- [x] Async background subagents, worktree isolation for parallel coding agents.
- [x] How the layers compose in one harness.

## Deep Study Notes

### Planner/generator/evaluator loops

- The pattern: planner decomposes the task; generator produces the artifact;
  evaluator (LLM judge and/or deterministic checker) scores against a rubric;
  generator revises with the feedback; loop until pass or budget exhausted.
- **When it pays** (Anthropic's criteria): output demonstrably improves when a
  human articulates feedback, and an LLM can produce that feedback. Strongest
  evaluators are hybrid: deterministic checks first (compiles, tests pass,
  schema-valid), LLM judge only for fuzzy criteria.
- **Termination**: use all of — max iterations (2-3; returns diminish fast),
  score threshold, no-improvement plateau detection, token budget. Return
  best-so-far, not last attempt.
- **Failure modes** (interview gold): *evaluator rubber-stamping* (judges are
  sycophantic — mitigate with rubric scores, required defect citations, a
  different model/persona for the judge); *oscillating revisions* (fix A
  breaks B — keep cumulative feedback history in the revision prompt);
  *reward hacking* the rubric's letter; *uncalibrated evaluator* stricter than
  achievable (always exhausts budget).

### Typed state graphs with checkpointed persistence

- **LangGraph** is canonical: `StateGraph` over a typed schema (TypedDict/
  Pydantic); nodes return *partial* updates; **reducers** (e.g.
  `Annotated[list, add_messages]`) define how updates merge — this is what
  makes parallel branches safe (append, don't clobber). `Send` does dynamic
  map-reduce fan-out.
- **Checkpointing**: compile with a checkpointer (Postgres in prod); state is
  snapshotted every superstep, keyed by `thread_id`. Buys: crash recovery
  (resume, don't replay), human-in-the-loop (`interrupt()` pauses mid-node,
  `Command(resume=...)` continues days later on another machine), time-travel
  debugging (fork any historical checkpoint with modified state), and a full
  audit trail.
- **Why typed state matters**: the state IS the API between agents — typing it
  is the same discipline as typing microservice contracts; schema validation
  catches drift at node boundaries instead of silent prompt corruption.
- **The caveat to volunteer**: checkpoints are save points at node
  granularity, *not* durable execution — a half-sent email inside a node can
  re-send on replay. **Temporal** gives true durable execution (deterministic
  workflow code, side effects in activities with recorded results, never
  repeats a completed tool call). Framing: LangGraph = agent-native state
  machine with save points; Temporal = workflow-native durability you put
  agents inside; teams run LangGraph inside Temporal activities.

### MCP and Brain/Hands/Session decoupling

- **Status 2026**: MCP is the de facto tool-integration standard; OpenAI and
  Google adopted it. Key spec waves: 2025-03 (Streamable HTTP, OAuth 2.1),
  2025-06 (elicitation, structured tool output), 2025-11 (async tasks,
  URL-mode elicitation where the client never sees credentials, extensions).
- **Primitives**: server→client: tools, resources, prompts. Client→server:
  *sampling* (server borrows the client's LLM — capability flows through one
  governed choke point), *roots* (scoped filesystem/URI access), *elicitation*
  (structured human input mid-operation).
- **The decoupling**:
  - **Brain** = the reasoning loop (MCP client/host). Decides *what*. Model
    choice and prompting live here — swappable without touching tools.
  - **Hands** = MCP tool servers. Decide *how*. Independently deployable
    microservices with their own credentials, scaling, versioning, sandboxing.
  - **Session** = conversation/state layer (checkpointer, thread store),
    separate from both: any brain resumes any session; hands stay stateless.
- **Security**: explicit trust boundaries (blast radius limited to a server's
  scope; credentials server-side; roots scope file access) but new attack
  surface: prompt-injected tool descriptions/results, tool poisoning, "rug
  pulls" (server changes behavior post-approval) — hence enterprise gateway/
  registry patterns with allow-listed, scanned servers.

### Isolated parallel agents and background subagents

- **Anthropic's multi-agent research system**: lead agent (frontier model)
  plans and spawns 3-5 subagents (mid-tier) in parallel, each with an explicit
  mandate (objective, output format, tool guidance, effort budget), then
  synthesizes. Result: **90.2% improvement** over single-agent on their
  research eval; ~80% of variance explained by token spend.
- **Context isolation is the point**: each subagent gets a fresh context, so
  exploration garbage never pollutes the orchestrator, total working context
  exceeds any single window, and agents reason without each other's biases.
  Corollary rule: **communicate via structured, compressed results — never
  shared raw context.** The subagent's final message is the interface.
- **Token economics**: chat ~1x, single agent ~4x, **multi-agent ~15x**.
  Parallelism pays only for parallelizable, high-value work (breadth-first
  research, multi-file audits) — not tightly-coupled sequential coding.
- **Async background subagents**: outlive a turn; parent continues working and
  is re-invoked on completion. Needs result mailboxes, continuation messaging,
  cancellation. Hard parts: long-running agents compound errors and need
  resume-from-failure, not restart.
- **Worktree isolation**: parallel coding agents each get a git worktree
  (separate working dir + branch, shared object DB) so they cannot clobber
  each other; fan-in becomes a git merge — standard tooling, reviewable diffs.
  Heavier: per-agent containers with scoped credentials.

### The composition

```
typed state graph (spine): plan -> fan-out -> synthesize -> evaluate -> done
    nodes: some are single LLM calls, some spawn isolated subagents
    edges: evaluator gates (deterministic checks + judge) between stages,
           fail routes back to generator with feedback appended to state
    tools: every tool call crosses an MCP boundary, per-agent capability scoping
    durability: every superstep checkpointed by thread_id; approvals interrupt/resume
```
Gate decisions are themselves checkpointed — an auditable trail of *why* the
system accepted each artifact.

## Common Failure Modes

- Evaluator rubber-stamps everything → the loop is cost without quality; require cited defects and rubric scores.
- No reducer on shared state → parallel branches clobber each other's writes.
- Side-effectful node replayed after crash → duplicate emails/charges; idempotency keys or Temporal-style activities.
- Subagents sharing raw context → token blowup and cross-contaminated reasoning.
- Vague subagent mandates → duplicate work and over-spawning (Anthropic's early failure).
- God-process with every credential → one prompt injection owns everything; MCP scoping exists for this.

## Implementation Notes

- Start with a workflow (explicit graph), not an open-ended agent; add autonomy only where the graph can't express the branching.
- Put iteration counters and feedback history *in the typed state*, not in prompts alone — they must survive resume.
- Scope each subagent's MCP servers to its mandate (research agent: search only; coding agent: repo + shell in its worktree).
- Log every gate decision with the evaluator's rubric scores; that is your audit trail.

In [ ]:
"""A miniature typed-state-graph harness with checkpointing and an
evaluator gate. Pure Python — the structure mirrors LangGraph exactly:
typed state, nodes returning partial updates, a reducer, a checkpointer
keyed by thread_id, and a bounded generate->evaluate loop.
"""
import json

CHECKPOINTS = {}  # thread_id -> list of state snapshots (prod: Postgres)


def checkpoint(thread_id, node, state):
    CHECKPOINTS.setdefault(thread_id, []).append(
        {"node": node, "state": json.loads(json.dumps(state))})


def merge(state, update):
    """Reducer: 'feedback' appends (safe under fan-in); others overwrite."""
    out = {**state}
    for k, v in update.items():
        out[k] = out.get(k, []) + v if k == "feedback" else v
    return out


# --- nodes: each returns a PARTIAL state update ---------------------------
def planner(state):
    return {"plan": ["draft summary", "cite evidence"]}


def generator(state):
    # Improves with accumulated feedback (simulates revision)
    quality = 0.5 + 0.25 * len(state["feedback"])
    return {"draft": f"summary v{state['iteration'] + 1}", "quality": quality,
            "iteration": state["iteration"] + 1}


def evaluator(state):
    """Hybrid gate: deterministic check + rubric score with cited defect."""
    if state["quality"] >= 0.9:
        return {"verdict": "PASS"}
    return {"verdict": "REVISE",
            "feedback": [f"v{state['iteration']}: evidence citations missing"]}


def run(thread_id, max_iterations=3):
    state = {"iteration": 0, "feedback": [], "verdict": None}
    state = merge(state, planner(state))
    checkpoint(thread_id, "planner", state)
    while state["iteration"] < max_iterations:
        state = merge(state, generator(state))
        checkpoint(thread_id, "generator", state)
        state = merge(state, evaluator(state))
        checkpoint(thread_id, "evaluator", state)
        if state["verdict"] == "PASS":
            break
    return state


final = run("thread-42")
print(f"verdict={final['verdict']} after {final['iteration']} iterations")
print("feedback history:", final["feedback"])
print(f"checkpoints persisted: {len(CHECKPOINTS['thread-42'])} "
      "(any of these can be resumed or forked)")
# time-travel: inspect state as it was after the first evaluation
print("state after first gate:", CHECKPOINTS["thread-42"][2]["state"]["verdict"])


## Practice

1. Add a `budget_tokens` field to the state and a second termination
   condition; show that the loop returns best-so-far when budget exhausts first.
2. Simulate rubber-stamping: make the evaluator PASS everything and explain
   which two mitigations from the notes you'd apply first.
3. Add a fan-out step: two parallel "research" nodes both append to a
   `findings` list via the reducer. What breaks if `findings` used overwrite
   semantics instead?
4. Whiteboard Brain/Hands/Session for a coding agent: name what lives in each
   box, and trace where a GitHub credential lives and why.
5. Explain the checkpoint-vs-durable-execution caveat with the half-sent
   email example, and when you'd reach for Temporal.

## Design Checklist

- [ ] Control flow is an explicit typed graph; autonomy only where needed.
- [ ] Every shared-state channel has explicit merge semantics (reducers).
- [ ] Checkpointer in prod (Postgres); thread_id resume tested, including mid-interrupt.
- [ ] Side-effectful steps are idempotent or activity-wrapped.
- [ ] Evaluator gates are hybrid (deterministic first, judge second) with cited defects.
- [ ] Loops bounded by iterations AND budget; best-so-far returned.
- [ ] Subagents: explicit mandates, structured results, own contexts, scoped MCP servers.
- [ ] Gate decisions and rubric scores logged for audit.